In [ ]:
# ============================================================
# 环境配置
# - Colab 用户：取消注释下方 Colab 区块
# - 本地 Jupyter 用户：直接运行 Local 区块
# ============================================================

# ── Colab 环境（取消注释后运行） ──
# !pip install torch torchvision matplotlib numpy -q

# ── 本地 Jupyter 环境 ──
import importlib
import subprocess
import sys

def _install(pkg: str):
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for _pkg in ['torch', 'torchvision', 'matplotlib', 'numpy']:
    _install(_pkg)

print('Environment ready.')


In [ ]:
import copy
import math
import time

import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torchvision
from torchvision.models import resnet18
import matplotlib.pyplot as plt

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'torch={torch.__version__}, torchvision={torchvision.__version__}')


# 视频理解经典方法代码实战：从 DeepVideo 到 Two-Stream，再到 Early Fusion

本 Notebook 对应三篇经典工作：

- *Large-scale Video Classification with Convolutional Neural Networks* (Karpathy et al., CVPR 2014)
- *Two-Stream Convolutional Networks for Action Recognition in Videos* (Simonyan & Zisserman, NeurIPS 2014)
- *Convolutional Two-Stream Network Fusion for Video Action Recognition* (Feichtenhofer et al., CVPR 2016)

我们用一个 **可解释、可在 Colab 运行的 toy video 任务**，把“视频分类为什么要建模时间信息”讲清楚。

| | 学习路径 (Learning) | 工程路径 (Engineering) |
|---|---|---|
| 目标 | 理解视频理解早期方法的关键设计 | 学会把论文思想翻译成工程模块 |
| 实现方式 | 手写 DeepVideo / Two-Stream / Fusion 核心模块 | 基于 `torchvision.models.resnet18` 组织双流网络 |
| 主训练目标 | `SingleFrame` 基线 + `TwoStreamLearningNet` | `TwoStreamResNet` |
| 适合场景 | 学原理、准备面试、能讲清方法演进 | 做原型、快速验证、连接真实项目 |
| 本 Notebook 的重点 | “为什么需要时间信息、怎么融合” | “同一思想如何写成可维护代码” |

> 这不是对原论文数据集的复现实验，而是一个 **从学习到简单工程** 的教学 notebook。


## 一、任务背景与最小必要理论

这三篇论文可以看成同一条主线上的连续提问：

1. **DeepVideo**：如果我把多帧直接塞给 CNN，会发生什么？
2. **Two-Stream**：如果 RGB 不够表达运动，我能不能把“外观”和“运动”分开建模？
3. **Early Fusion**：如果已经有两条流，那么应该**怎么融合、在哪融合、时间上怎么融合**？

本 Notebook 的任务是一个简化版 **视频分类 / 动作识别**：
- 输入：一段短视频
- 输出：动作类别（left / right / up / down / stationary）

为了保证 Colab 可运行，我们不用 UCF-101 / HMDB-51，而是构造一个 **moving digits** toy dataset：
- 每个视频里只有一个 MNIST 数字在移动
- 动作标签只由“运动方向”决定
- 我们特意让**中间帧位置与方向无关**，避免单帧基线只靠位置泄漏猜答案

### DeepVideo 的四种融合思路

- **Single Frame**：只看一帧，完全不建模时间
- **Late Fusion**：两帧各自过 CNN，最后再合并决策
- **Early Fusion**：把多帧在通道维拼起来，尽早让网络看到时间上下文
- **Slow Fusion**：先局部融合，再逐步聚合成视频级特征

### Two-Stream 的核心思想

- 空间流负责 appearance，也就是“看到什么”
- 时间流负责 motion，也就是“怎么动”
- 最简单的分数级融合可以写成：`fused = alpha * spatial_logits + (1 - alpha) * temporal_logits`

### Early Fusion 的空间融合

- **Sum Fusion**：直接相加
- **Max Fusion**：逐元素取最大
- **Concatenation Fusion**：通道拼接
- **Conv Fusion**：拼接后再过可学习卷积
- **Bilinear Fusion**：更强的交互；这里用逐元素乘积做轻量近似演示

### 本 Notebook 的现实化简

真实 Two-Stream 常用 **dense optical flow**。为了 Colab 友好，这里用：

`frame_diff[t] = frame[t+1] - frame[t]`

来近似“运动信息”。它不等于真正光流，但足够帮助我们学习两条流在代码中的职责分工。


## 二、组件映射表（必看）

| 论文组件 | 学习路径覆盖 | 工程路径 / API 对应 | 状态 |
|---|---|---|---|
| Single Frame | `SingleFrameModel` | 单路 `ResNetStream` | Runnable |
| Late Fusion | `LateFusionModel` | logits 加权平均 | Runnable |
| Early Fusion | `DeepVideoEarlyFusion` | stacked-channel input | Runnable |
| Slow Fusion | `SlowFusionModel` | staged feature grouping | Runnable |
| Spatial Stream | `TwoStreamLearningNet.spatial` | `TwoStreamResNet.spatial_stream` | Runnable |
| Temporal Stream | frame diff stack + `temporal` branch | `TwoStreamResNet.temporal_stream` | Runnable |
| Score-level fusion | `alpha * s + (1 - alpha) * t` | 同一公式保留 | Runnable |
| Spatial Fusion (sum/max/cat/conv/bilinear) | 独立 fusion modules | 真实工程中常体现在 feature fusion block | Runnable |
| Temporal Fusion (2D / 3D / 3D conv+pool) | 小型张量 demo | 3D video backbones / temporal heads | Illustrative |
| Dense optical flow extraction | 仅解释，不做 TV-L1 / RAFT | 真实系统通常作为预处理 | Explain-only |
| UCF-101 / HMDB-51 全量复现 | 仅讲方法，不做大规模训练 | 真实复现需更大算力和更长训练 | Explain-only |


In [ ]:
CFG = {
    'n_samples': 300,
    'n_frames': 9,
    'canvas_size': 48,
    'digit_size': 28,
    'step_pixels': 1,
    'n_classes': 5,
    'train_ratio': 0.8,
    'batch_size': 32,
    'lr': 1e-3,
    'epochs_baseline': 2,
    'epochs_learning': 3,
    'epochs_engineering': 2,
    'spatial_weight': 0.5,
}
ACTION_NAMES = ['left', 'right', 'up', 'down', 'stationary']


class MovingDigitsVideoDataset(Dataset):
    DIRECTIONS = {
        0: (-1, 0),
        1: (1, 0),
        2: (0, -1),
        3: (0, 1),
        4: (0, 0),
    }

    def __init__(self, n_samples, n_frames, canvas_size, digit_size=28, step_pixels=1, seed=42):
        super().__init__()
        self.n_frames = n_frames
        self.canvas_size = canvas_size
        self.digit_size = digit_size
        self.step_pixels = step_pixels

        mnist = torchvision.datasets.MNIST(root='./data', train=True, download=True)
        digit_bank = mnist.data.float() / 255.0

        rng = np.random.default_rng(seed)
        mid = n_frames // 2
        max_shift = step_pixels * mid

        x_min = max_shift
        x_max = canvas_size - digit_size - max_shift
        y_min = max_shift
        y_max = canvas_size - digit_size - max_shift
        if x_max < x_min or y_max < y_min:
            raise ValueError('Canvas is too small for the chosen trajectory setup.')

        videos, labels = [], []
        for i in range(n_samples):
            action_id = i % len(ACTION_NAMES)
            dx, dy = self.DIRECTIONS[action_id]
            digit = digit_bank[int(rng.integers(0, len(digit_bank)))]

            x_mid = int(rng.integers(x_min, x_max + 1))
            y_mid = int(rng.integers(y_min, y_max + 1))

            frames = []
            for t in range(n_frames):
                offset = t - mid
                x = x_mid + dx * step_pixels * offset
                y = y_mid + dy * step_pixels * offset
                canvas = torch.zeros(canvas_size, canvas_size)
                canvas[y:y + digit_size, x:x + digit_size] = digit
                frames.append(canvas)

            videos.append(torch.stack(frames))
            labels.append(action_id)

        self.videos = torch.stack(videos)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.videos[idx], self.labels[idx]


def compute_frame_diffs(videos):
    return videos[:, 1:] - videos[:, :-1]


class TwoStreamDataset(Dataset):
    def __init__(self, videos, labels):
        self.videos = videos
        self.diffs = compute_frame_diffs(videos)
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        mid = self.videos.shape[1] // 2
        spatial = self.videos[idx, mid].unsqueeze(0)
        temporal = self.diffs[idx]
        return spatial, temporal, self.labels[idx]


dataset = MovingDigitsVideoDataset(
    n_samples=CFG['n_samples'],
    n_frames=CFG['n_frames'],
    canvas_size=CFG['canvas_size'],
    digit_size=CFG['digit_size'],
    step_pixels=CFG['step_pixels'],
    seed=SEED,
)

g = torch.Generator().manual_seed(SEED)
perm = torch.randperm(len(dataset), generator=g)
n_train = int(len(dataset) * CFG['train_ratio'])
train_idx = perm[:n_train]
test_idx = perm[n_train:]

train_videos = dataset.videos[train_idx]
test_videos = dataset.videos[test_idx]
train_labels = dataset.labels[train_idx]
test_labels = dataset.labels[test_idx]

train_videos_5d = train_videos.unsqueeze(1)
test_videos_5d = test_videos.unsqueeze(1)

video_train_ds = TensorDataset(train_videos_5d, train_labels)
video_test_ds = TensorDataset(test_videos_5d, test_labels)
two_stream_train_ds = TwoStreamDataset(train_videos, train_labels)
two_stream_test_ds = TwoStreamDataset(test_videos, test_labels)

pin_memory = torch.cuda.is_available()

video_train_loader = DataLoader(video_train_ds, batch_size=CFG['batch_size'], shuffle=True, pin_memory=pin_memory)
video_test_loader = DataLoader(video_test_ds, batch_size=CFG['batch_size'], shuffle=False, pin_memory=pin_memory)
two_stream_train_loader = DataLoader(two_stream_train_ds, batch_size=CFG['batch_size'], shuffle=True, pin_memory=pin_memory)
two_stream_test_loader = DataLoader(two_stream_test_ds, batch_size=CFG['batch_size'], shuffle=False, pin_memory=pin_memory)

print('Config:', CFG)
print('Dataset videos shape:', tuple(dataset.videos.shape))
print('Dataset labels shape:', tuple(dataset.labels.shape))
print('Class counts:', torch.bincount(dataset.labels).tolist())


In [ ]:
fig, axes = plt.subplots(len(ACTION_NAMES), 4, figsize=(10, 12))

for action_id, action_name in enumerate(ACTION_NAMES):
    idx = int((dataset.labels == action_id).nonzero(as_tuple=True)[0][0])
    video = dataset.videos[idx]
    diffs = compute_frame_diffs(video.unsqueeze(0)).squeeze(0)

    frame_ids = [0, CFG['n_frames'] // 2, CFG['n_frames'] - 1]
    titles = ['Frame 0', 'Middle frame', 'Last frame']

    for col, frame_id in enumerate(frame_ids):
        axes[action_id, col].imshow(video[frame_id], cmap='gray', vmin=0, vmax=1)
        axes[action_id, col].set_title(titles[col], fontsize=9)
        axes[action_id, col].set_xticks([])
        axes[action_id, col].set_yticks([])

    axes[action_id, 3].imshow(diffs[CFG['n_frames'] // 2 - 1], cmap='RdBu_r', vmin=-1, vmax=1)
    axes[action_id, 3].set_title('Frame diff', fontsize=9)
    axes[action_id, 3].set_xticks([])
    axes[action_id, 3].set_yticks([])
    axes[action_id, 0].set_ylabel(action_name, fontsize=10)

fig.suptitle('Toy videos for each action class', fontsize=14)
plt.tight_layout()
plt.show()

video_batch, label_batch = next(iter(video_train_loader))
spatial_batch, temporal_batch, ts_label_batch = next(iter(two_stream_train_loader))

print('Video batch shape   :', tuple(video_batch.shape))
print('Spatial batch shape :', tuple(spatial_batch.shape))
print('Temporal batch shape:', tuple(temporal_batch.shape))
print('Label batch shape   :', tuple(label_batch.shape))


## 三、共享组件：训练、评估、画图

下面这部分是两条路径共用的基础设施：
- 统一训练循环
- 统一准确率评估
- 统一训练曲线绘制

后面你要重点关注的不是“训练函数怎么写”，而是：
- 输入张量如何组织
- 模型内部如何分工
- 融合发生在什么层级


In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def evaluate_accuracy(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in loader:
            *inputs, labels = [b.to(device) for b in batch]
            logits = model(*inputs)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total


def train_classifier(model, train_loader, test_loader, epochs, lr):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    history = {'train_loss': [], 'train_acc': [], 'test_acc': []}
    best_acc = 0.0
    best_state = copy.deepcopy(model.state_dict())

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for batch in train_loader:
            *inputs, labels = [b.to(device) for b in batch]
            optimizer.zero_grad()
            logits = model(*inputs)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_acc = correct / total
        test_acc = evaluate_accuracy(model, test_loader)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)

        if test_acc >= best_acc:
            best_acc = test_acc
            best_state = copy.deepcopy(model.state_dict())

        print(f'Epoch {epoch:02d}/{epochs} | Train loss={train_loss:.4f} | Train acc={train_acc:.3f} | Test acc={test_acc:.3f}')

    model.load_state_dict(best_state)
    history['best_test_acc'] = best_acc
    return history


def plot_history(history, title_prefix):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    axes[0].plot(history['train_loss'], marker='o')
    axes[0].set_title(f'{title_prefix} loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')

    axes[1].plot(history['train_acc'], marker='o', label='Train')
    axes[1].plot(history['test_acc'], marker='s', label='Test')
    axes[1].set_title(f'{title_prefix} accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()

    plt.tight_layout()
    plt.show()


## 四、学习路径：先把“时间信息怎么进模型”讲清楚

学习路径分三层：

1. **DeepVideo 家族**：看四种“把多帧送进 CNN”的方式
2. **Two-Stream**：把外观和运动拆开建模
3. **Early Fusion 思想**：研究两条流到底该怎么融合

这里的策略是：
- **真正训练**：`SingleFrameModel` 与 `TwoStreamLearningNet`
- **可运行演示**：`LateFusionModel`、`DeepVideoEarlyFusion`、`SlowFusionModel`、各类 fusion module

因为你的目标是“会学、会讲、能面试”，所以我们优先训练**最有解释力**的模型，而不是把所有变体都堆成长时间实验。


In [ ]:
class Small2DBackbone(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
        )

    def forward(self, x):
        return self.net(x)


class SingleFrameModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.encoder = Small2DBackbone(in_channels=1)
        self.head = nn.Linear(128, num_classes)

    def forward(self, video):
        mid = video.shape[2] // 2
        frame = video[:, :, mid]
        feat = self.encoder(frame)
        return self.head(feat)


class LateFusionModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.encoder = Small2DBackbone(in_channels=1)
        self.head = nn.Linear(128 * 2, num_classes)

    def forward(self, video):
        first_frame = video[:, :, 0]
        last_frame = video[:, :, -1]
        feat_first = self.encoder(first_frame)
        feat_last = self.encoder(last_frame)
        fused = torch.cat([feat_first, feat_last], dim=1)
        return self.head(fused)


class DeepVideoEarlyFusion(nn.Module):
    def __init__(self, num_frames, num_classes):
        super().__init__()
        self.encoder = Small2DBackbone(in_channels=num_frames)
        self.head = nn.Linear(128, num_classes)

    def forward(self, video):
        x = video.squeeze(1)
        feat = self.encoder(x)
        return self.head(feat)


class SlowFusionModel(nn.Module):
    def __init__(self, num_frames, num_classes):
        super().__init__()
        assert num_frames % 3 == 0, 'This demo assumes 3 equal frame groups.'
        self.group_size = num_frames // 3
        self.clip_encoder = Small2DBackbone(in_channels=self.group_size)
        self.fuse12 = nn.Linear(128 * 2, 128)
        self.fuse123 = nn.Linear(128 * 2, 128)
        self.head = nn.Linear(128, num_classes)

    def forward(self, video):
        b, c, t, h, w = video.shape
        clips = []
        for start in range(0, t, self.group_size):
            clip = video[:, :, start:start + self.group_size]
            clip = clip.reshape(b, c * self.group_size, h, w)
            clips.append(self.clip_encoder(clip))

        fused12 = F.relu(self.fuse12(torch.cat([clips[0], clips[1]], dim=1)))
        fused123 = F.relu(self.fuse123(torch.cat([fused12, clips[2]], dim=1)))
        return self.head(fused123)


class TwoStreamLearningNet(nn.Module):
    def __init__(self, temporal_channels, num_classes, spatial_weight=0.5):
        super().__init__()
        self.spatial = Small2DBackbone(in_channels=1)
        self.temporal = Small2DBackbone(in_channels=temporal_channels)
        self.spatial_head = nn.Linear(128, num_classes)
        self.temporal_head = nn.Linear(128, num_classes)
        self.spatial_weight = spatial_weight

    def forward(self, spatial_input, temporal_input):
        spatial_logits = self.spatial_head(self.spatial(spatial_input))
        temporal_logits = self.temporal_head(self.temporal(temporal_input))
        fused_logits = self.spatial_weight * spatial_logits + (1 - self.spatial_weight) * temporal_logits
        return fused_logits


In [ ]:
demo_video = video_batch[:2]

demo_models = {
    'SingleFrame': SingleFrameModel(CFG['n_classes']),
    'LateFusion': LateFusionModel(CFG['n_classes']),
    'EarlyFusion': DeepVideoEarlyFusion(CFG['n_frames'], CFG['n_classes']),
    'SlowFusion': SlowFusionModel(CFG['n_frames'], CFG['n_classes']),
}

for name, model in demo_models.items():
    logits = model(demo_video)
    print(f'{name:12s} | output shape={tuple(logits.shape)} | params={count_parameters(model):,}')

learning_demo = TwoStreamLearningNet(CFG['n_frames'] - 1, CFG['n_classes'], CFG['spatial_weight'])
learning_logits = learning_demo(spatial_batch[:2], temporal_batch[:2])
print('TwoStreamLearningNet output shape:', tuple(learning_logits.shape))
print('TwoStreamLearningNet params:', f'{count_parameters(learning_demo):,}')


### 学习路径主训练目标

我们只重点训练两个模型：

1. **SingleFrame baseline**：证明“只看静态图像”是明显偏弱的基线
2. **TwoStreamLearningNet**：证明把外观流和运动流拆开后，模型更容易学到方向信息

这部分最适合你在面试里讲：

- 为什么 baseline 不够
- 为什么 motion 信息重要
- 为什么双流结构是一个自然改进


In [ ]:
baseline_model = SingleFrameModel(num_classes=CFG['n_classes'])
print('SingleFrame params:', f'{count_parameters(baseline_model):,}')
print('Training SingleFrame baseline...')

history_baseline = train_classifier(
    baseline_model,
    video_train_loader,
    video_test_loader,
    epochs=CFG['epochs_baseline'],
    lr=CFG['lr'],
)

plot_history(history_baseline, 'SingleFrame baseline')
print('Best test accuracy:', f"{history_baseline['best_test_acc']:.3f}")


In [ ]:
learning_model = TwoStreamLearningNet(
    temporal_channels=CFG['n_frames'] - 1,
    num_classes=CFG['n_classes'],
    spatial_weight=CFG['spatial_weight'],
)
print('Learning-path Two-Stream params:', f'{count_parameters(learning_model):,}')
print('Training learning-path Two-Stream...')

history_learning = train_classifier(
    learning_model,
    two_stream_train_loader,
    two_stream_test_loader,
    epochs=CFG['epochs_learning'],
    lr=CFG['lr'],
)

plot_history(history_learning, 'Learning-path Two-Stream')
print('Best test accuracy:', f"{history_learning['best_test_acc']:.3f}")


In [ ]:
print(f"{'Model':<28s} {'Params':>12s} {'Best test acc':>14s}")
print('-' * 58)
print(f"{'SingleFrame baseline':<28s} {count_parameters(baseline_model):>12,d} {history_baseline['best_test_acc']:>14.3f}")
print(f"{'TwoStreamLearningNet':<28s} {count_parameters(learning_model):>12,d} {history_learning['best_test_acc']:>14.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(history_baseline['test_acc'], marker='o', label='SingleFrame')
axes[0].plot(history_learning['test_acc'], marker='s', label='TwoStream')
axes[0].set_title('Baseline vs two-stream test accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(history_baseline['train_loss'], marker='o', label='SingleFrame')
axes[1].plot(history_learning['train_loss'], marker='s', label='TwoStream')
axes[1].set_title('Baseline vs two-stream train loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.show()

print('Interpretation: the two-stream model should beat the single-frame baseline because direction lives in motion rather than in any single image.')


## 五、Early Fusion：两条流到底怎么融合？

CVPR 2016 的重点不是再发明“两条流”，而是进一步问：

1. 两条流的特征应该怎么融合？
2. 融合发生在网络的哪一层更合理？
3. 时间维上应该怎样继续聚合？

下面先做**空间融合**的小型可运行演示，再做一个**时间融合**的 shape demo。


In [ ]:
class SmallFeatureExtractor(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class SpatialFusionSum(nn.Module):
    def forward(self, a, b):
        return a + b


class SpatialFusionMax(nn.Module):
    def forward(self, a, b):
        return torch.maximum(a, b)


class SpatialFusionCat(nn.Module):
    def forward(self, a, b):
        return torch.cat([a, b], dim=1)


class SpatialFusionConv(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels * 2, channels, kernel_size=1)

    def forward(self, a, b):
        x = torch.cat([a, b], dim=1)
        return self.conv(x)


class SpatialFusionBilinear(nn.Module):
    def forward(self, a, b):
        return a * b


spatial_feature_extractor = SmallFeatureExtractor(in_channels=1)
temporal_feature_extractor = SmallFeatureExtractor(in_channels=CFG['n_frames'] - 1)

spatial_features = spatial_feature_extractor(spatial_batch[:2])
temporal_features = temporal_feature_extractor(temporal_batch[:2])

fusion_modules = {
    'Sum': SpatialFusionSum(),
    'Max': SpatialFusionMax(),
    'Cat': SpatialFusionCat(),
    'Conv': SpatialFusionConv(channels=64),
    'Bilinear': SpatialFusionBilinear(),
}

print('Spatial feature shape :', tuple(spatial_features.shape))
print('Temporal feature shape:', tuple(temporal_features.shape))
for name, module in fusion_modules.items():
    out = module(spatial_features, temporal_features)
    print(f'{name:10s} -> {tuple(out.shape)}')

feature_volume = torch.randn(2, 64, 5, 8, 8)
pooled_2d = feature_volume.mean(dim=2)
pool3d = nn.MaxPool3d(kernel_size=(2, 1, 1), stride=(2, 1, 1))
after_3d_pool = pool3d(feature_volume)
conv3d = nn.Conv3d(64, 64, kernel_size=(3, 1, 1), padding=(1, 0, 0))
after_3d_conv = F.relu(conv3d(feature_volume))
after_3d_conv_pool = pool3d(after_3d_conv)

print('Input feature volume     :', tuple(feature_volume.shape))
print('After 2D temporal mean   :', tuple(pooled_2d.shape))
print('After 3D pooling         :', tuple(after_3d_pool.shape))
print('After 3D conv + 3D pool  :', tuple(after_3d_conv_pool.shape))


## 六、工程路径：用现成 backbone 组织一个更像项目代码的双流网络

在实际项目里，你通常不会从零手写所有卷积层，而是会：

1. 复用成熟 backbone
2. 改输入层适配自己的数据
3. 保留清晰的模块边界
4. 用尽可能少的代码表达同一思想

这里我们用 `torchvision.models.resnet18(weights=None)` 做一个 **简单工程版 Two-Stream**：
- 空间流：吃中间帧（1 channel）
- 时间流：吃帧差分堆叠（`T-1` channels）
- 最后仍然做 logits 级融合

> 之所以不用预训练权重，是因为我们的 toy 数据是灰度数字 + 帧差分，不是标准 RGB 自然图像。


In [ ]:
class ResNetStream(nn.Module):
    def __init__(self, in_channels, num_classes):
        super().__init__()
        backbone = resnet18(weights=None)
        backbone.conv1 = nn.Conv2d(
            in_channels,
            64,
            kernel_size=7,
            stride=2,
            padding=3,
            bias=False,
        )
        backbone.fc = nn.Linear(backbone.fc.in_features, num_classes)
        self.backbone = backbone

    def forward(self, x):
        return self.backbone(x)


class TwoStreamResNet(nn.Module):
    def __init__(self, temporal_channels, num_classes, spatial_weight=0.5):
        super().__init__()
        self.spatial_stream = ResNetStream(in_channels=1, num_classes=num_classes)
        self.temporal_stream = ResNetStream(in_channels=temporal_channels, num_classes=num_classes)
        self.spatial_weight = spatial_weight

    def forward(self, spatial_input, temporal_input):
        spatial_logits = self.spatial_stream(spatial_input)
        temporal_logits = self.temporal_stream(temporal_input)
        fused_logits = self.spatial_weight * spatial_logits + (1 - self.spatial_weight) * temporal_logits
        return fused_logits


engineering_model = TwoStreamResNet(
    temporal_channels=CFG['n_frames'] - 1,
    num_classes=CFG['n_classes'],
    spatial_weight=CFG['spatial_weight'],
)

eng_logits = engineering_model(spatial_batch[:2], temporal_batch[:2])
print('TwoStreamResNet output shape:', tuple(eng_logits.shape))
print('TwoStreamResNet params:', f'{count_parameters(engineering_model):,}')

print('Training engineering-path Two-Stream ResNet...')
history_engineering = train_classifier(
    engineering_model,
    two_stream_train_loader,
    two_stream_test_loader,
    epochs=CFG['epochs_engineering'],
    lr=CFG['lr'],
)

plot_history(history_engineering, 'Engineering-path Two-Stream ResNet')
print('Best test accuracy:', f"{history_engineering['best_test_acc']:.3f}")


## 七、学习路径 vs 工程路径对比

| 对比维度 | 学习路径 | 工程路径 |
|---|---|---|
| 教育价值 | 你能看到单帧、Late / Early / Slow Fusion、Two-Stream 的设计差异 | 你能看到同一思想如何压缩成少量、可维护的模块 |
| 代码量 | 较多：为了讲清组件职责，需要显式写出结构 | 较少：大量细节被 backbone 和库 API 隐藏 |
| 灵活性 | 高：你能随时改 fusion 位置、分支结构、输入组织 | 中：容易改头和输入层，但不适合频繁改底层细节 |
| 稳定性 | 中：适合教学，边界条件和性能优化都要自己管 | 高：模块成熟，接口清晰，更接近真实项目写法 |
| 工业适配度 | 适合研究原型、方法理解、面试讲解 | 适合快速原型、实验脚手架、项目代码起点 |
| 适用场景 | 学原理；解释论文；准备实习面试 | 快速做 demo；把想法接进现有训练流程 |

> 一句话总结：**学习路径帮你回答“为什么这么设计”，工程路径帮你回答“项目里怎么写出来”。**


In [ ]:
print(f"{'Model':<30s} {'Params':>12s} {'Best test acc':>14s}")
print('-' * 62)
print(f"{'Learning: TwoStreamLearningNet':<30s} {count_parameters(learning_model):>12,d} {history_learning['best_test_acc']:>14.3f}")
print(f"{'Engineering: TwoStreamResNet':<30s} {count_parameters(engineering_model):>12,d} {history_engineering['best_test_acc']:>14.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(history_learning['test_acc'], marker='o', label='Learning path')
axes[0].plot(history_engineering['test_acc'], marker='s', label='Engineering path')
axes[0].set_title('Learning vs engineering test accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(history_learning['train_loss'], marker='o', label='Learning path')
axes[1].plot(history_engineering['train_loss'], marker='s', label='Engineering path')
axes[1].set_title('Learning vs engineering train loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.show()


## 八、训练 vs 推理：这个任务里差异没有 Transformer 那么强

这类视频分类模型不像 seq2seq / LLM 那样存在 teacher forcing 或自回归解码，因此训练与推理的差异主要体现在：

| 阶段 | 学习路径行为 | 工程路径行为 |
|---|---|---|
| 训练 | `model.train()`；根据标签优化融合后的 logits | 同样使用标签监督，只是 backbone 细节被库封装 |
| 推理 | `model.eval()`；输入中间帧 + 帧差分，输出类别分数 | 同样调用 `forward()`，但特征提取由 ResNet backbone 完成 |
| 核心区别 | 重点在于“输入组织 + 融合方式” | 重点在于“模块复用 + 代码维护性” |

所以你在面试里可以这样概括：

> 这类早期视频分类方法，训练 / 推理的**算法范式差异不大**；真正重要的是模型怎样把时间信息纳入表示。


In [ ]:
def show_prediction_samples(learning_model, engineering_model, loader, action_names, n_examples=6):
    learning_model.eval()
    engineering_model.eval()

    spatial, temporal, labels = next(iter(loader))
    spatial = spatial[:n_examples].to(device)
    temporal = temporal[:n_examples].to(device)
    labels = labels[:n_examples].to(device)

    with torch.no_grad():
        learning_probs = learning_model(spatial, temporal).softmax(dim=1)
        engineering_probs = engineering_model(spatial, temporal).softmax(dim=1)

    cols = math.ceil(n_examples / 2)
    fig, axes = plt.subplots(2, cols, figsize=(3 * cols, 6))
    axes = np.array(axes).reshape(-1)

    for i in range(n_examples):
        axes[i].imshow(spatial[i, 0].cpu(), cmap='gray', vmin=0, vmax=1)
        true_name = action_names[labels[i].item()]
        learn_name = action_names[learning_probs[i].argmax().item()]
        eng_name = action_names[engineering_probs[i].argmax().item()]
        axes[i].set_title(
            f'True={true_name}\nL={learn_name} ({learning_probs[i].max().item():.2f})\nE={eng_name} ({engineering_probs[i].max().item():.2f})',
            fontsize=9,
        )
        axes[i].axis('off')

    for j in range(n_examples, len(axes)):
        axes[j].axis('off')

    fig.suptitle('Learning path vs engineering path predictions', fontsize=14)
    plt.tight_layout()
    plt.show()


show_prediction_samples(learning_model, engineering_model, two_stream_test_loader, ACTION_NAMES, n_examples=6)


## 九、结果与边界：这个 Notebook 能教会你什么，不能证明什么？

### 你能学到什么

1. **方法演进逻辑**：DeepVideo → Two-Stream → Early Fusion 的问题意识是什么
2. **输入张量组织**：视频 `(B, C, T, H, W)`、双流输入 `(spatial, temporal)` 怎么构造
3. **融合层级差异**：输入级、特征级、分数级融合各自代表什么抽象层级
4. **工程表达方式**：同样的两条流思想，怎样压缩成可维护的 ResNet 版本
5. **面试回答骨架**：为什么单帧不够、为什么要光流 / 时间流、为什么后来出现 3D CNN / Video Transformer

### 这份 Notebook 不能证明什么

1. **不能证明论文原始精度**：这里没有训练 UCF-101 / HMDB-51 / Kinetics
2. **不能证明某种 fusion 在真实数据上一定最优**：toy task 只用于帮助理解结构
3. **不能替代真正的视频工程**：真实系统还会涉及 clip sampling、光流提取、长视频切分、多机训练等问题

### 正确的使用方式

把这份 notebook 当成：
- **学习入口**：先把视频理解早期思想讲通
- **面试素材**：能从“问题—方法—限制—后续发展”四步说清楚
- **工程跳板**：知道如何把论文结构写成清晰的模块代码


## 十、面试高频题

### Q1：为什么单帧图像分类模型不适合做视频动作识别？

**核心答案**：因为很多动作类别的区分信息不在静态外观，而在时间变化。

- 单帧只能回答“画面里有什么”
- 动作识别常常需要回答“目标在怎么动”
- 例如 left / right 在某一张中间帧里可能完全一样，只有连续帧才有区别
- 这正是视频理解比图像理解更难的根本原因

### Q2：Two-Stream 为什么有效？

**核心答案**：它把“外观”和“运动”这两类信号拆开建模，降低了单一路径同时处理两种信息的难度。

- 空间流负责 appearance
- 时间流负责 motion
- 最后用分数级融合把两个视角合起来
- 这种分工让模型结构更清晰，也更容易分析每条分支在做什么

### Q3：帧差分和光流有什么区别？

**核心答案**：帧差分是最便宜的运动近似，光流是更精确的像素位移场。

- 帧差分：`I[t+1] - I[t]`，实现简单、计算快
- 光流：估计每个像素的运动向量 `(dx, dy)`，表达力更强
- 帧差分足够做教学 demo
- 真正的 Two-Stream 论文通常使用 dense optical flow

### Q4：Late Fusion、Early Fusion、Slow Fusion 的区别是什么？

- **Late Fusion**：先各自提特征 / 分数，最后再合并；实现简单，但跨时间交互较晚
- **Early Fusion**：尽早让网络看到多帧；时间信息进入得早，但通道数会变大
- **Slow Fusion**：先局部融合，再逐步汇总；在“信息交互深度”和“结构复杂度”之间做折中

面试时一句话概括：

> 它们的本质区别在于：**时间信息是在网络的哪一层进入、以什么粒度进入、什么时候开始发生交互。**

### Q5：为什么 Two-Stream 比 SingleFrame 更合理？

- SingleFrame 假设“静态外观足够区分类别”
- Two-Stream 显式承认“很多类别需要运动信息”
- 因此 Two-Stream 更符合视频任务的信号结构
- 这也是它在历史上成为经典范式的原因

### Q6：为什么 Conv Fusion 往往比 Sum / Max / Cat 更强？

**核心答案**：因为 Conv Fusion 是可学习的融合。

- Sum / Max 是固定规则
- Cat 只是把信息摆在一起，后续还要自己学怎么用
- Conv Fusion 在拼接后立刻做一次可学习变换，可以自动学习“该保留什么、抑制什么”
- 所以它通常是更平衡的特征融合方案

### Q7：3D Conv + 3D Pooling 为什么比简单时间平均更有表现力？

- 时间平均只是在聚合，几乎没有“学习时间模式”的能力
- 3D Pooling 把时间维也纳入操作对象，但本身仍是固定规则
- 3D Conv + 3D Pooling 可以显式学习时空联合模式
- 所以后来的 I3D、SlowFast、R(2+1)D 都在不同程度上强化了这一方向

### Q8：为什么后来很多方法逐渐减少对光流的依赖？

- 光流提取代价高，难做实时系统
- 双流需要维护两套输入与预处理链路，工程成本大
- 更强的 3D CNN / Transformer 在大数据下可以直接从 RGB 学到更强的时空表示
- 所以研究趋势逐步从“手工显式运动特征”转向“端到端时空建模”


## 十一、面试速答提纲 + 项目表达 + 延伸阅读

### 面试速答提纲

| # | 问题 | 一句话回答 |
|---|---|---|
| 1 | 为什么视频理解比图像理解更难？ | 因为很多类别差异存在于时间变化，而不是单帧外观。 |
| 2 | Two-Stream 的核心思想是什么？ | 把 appearance 和 motion 拆成两条流分别建模，再做融合。 |
| 3 | 帧差分和光流的关系？ | 帧差分是廉价近似，光流是更精确的像素运动场。 |
| 4 | Late / Early / Slow Fusion 的区别？ | 区别在于时间信息进入网络的层级和交互发生的时机。 |
| 5 | 为什么 Conv Fusion 常更合理？ | 因为它是可学习融合，而不是固定规则。 |
| 6 | 为什么后续方法减少了光流依赖？ | 因为光流成本高，而 3D CNN / Transformer 能直接从 RGB 学时空表示。 |

### 项目表达模板

如果面试官问“你做过什么相关项目”，你可以这样讲：

> 我做了一个从学习到简单工程的视频理解 notebook。学习路径里，我从零实现了 DeepVideo 的几种融合方式、Two-Stream 的双流结构，以及 Early Fusion 的特征融合模块；工程路径里，我又用 `torchvision` 的 ResNet backbone 搭了一个简洁版双流模型。通过这个项目，我不仅能写出代码，更能解释为什么视频任务需要时间建模、不同 fusion 策略分别解决什么问题，以及这些经典方法为什么后来被 I3D、SlowFast、Video Transformer 逐步替代。

### 与后续方法的对比

| 方法 | 核心思想 | 相对本 Notebook 的升级点 | 面试时一句话 |
|---|---|---|---|
| DeepVideo | 把多帧直接塞进 CNN | 首次系统比较多种时空融合思路 | 证明“简单堆帧并不够” |
| Two-Stream | 外观流 + 运动流 | 显式引入 motion 分支 | 是视频理解早期最经典范式之一 |
| Early Fusion | 研究怎么融合、在哪融合 | 从“有两条流”继续推进到“如何更优地交互” | 关注融合层级，而不只是双流本身 |
| I3D | 把 2D CNN 膨胀成 3D CNN | 更强的端到端时空建模 | 3D CNN 代表作 |
| SlowFast | 快慢双路径建模不同时间尺度 | 更细粒度的时间分辨率分工 | 把“时间建模”做得更彻底 |
| TimeSformer / ViViT | 用 Transformer 做视频理解 | 全局时空注意力 | 视频版 Transformer 路线 |

### 进阶探索方向

1. **把 frame diff 替换成真正光流**：更接近原始 Two-Stream 论文
2. **把工程路径升级成 3D backbone**：例如 `torchvision.models.video.r3d_18`
3. **加入 clip sampling**：理解长视频如何切成多个 clip 再做聚合
4. **对比 I3D / SlowFast / TimeSformer**：把“经典 CNN 路线”与“现代视频模型”串起来
5. **尝试真实小数据集**：例如 UCF-101 的极小子集，观察 toy task 与真实任务的差异
